In [1]:
import sys
sys.path.append('../')
import scipy.io as sio
import mat73
import pandas as pd
import torch
import numpy as np
import torch.optim as optim
import torch.nn
import sklearn
import sklearn.metrics
import matplotlib.pyplot as plt
from alive_progress import alive_bar
from  utils.my_classes import dataset 
from torch.utils.data import DataLoader
import utils.DNN_functions as DNN_functions
import scipy
import random
import utils.AMSloss
from speechbrain.pretrained import SpeakerRecognition
import pickle
import os
from utils.my_classes import dataset
import utils.eval_metrics as eval_metrics
import copy
from speechbrain.pretrained import SpeakerRecognition

#To get my GPU device - GTX 4070 :)
seed = 42  # You can choose any integer value as the seed
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu');

if torch.cuda.is_available():
    print(torch.cuda.device_count())
    print(torch.cuda.device(0))
    print(torch.cuda.get_device_name(0))
    print(device)

The torchaudio backend is switched to 'soundfile'. Note that 'sox_io' is not supported on Windows.
c:\Users\avish\OneDrive\Desktop\thesis_research\avishai_work\env\lib\site-packages\speechbrain\utils\torch_audio_backend.py:22: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")
c:\Users\avish\OneDrive\Desktop\thesis_research\avishai_work\env\lib\site-packages\torch\onnx\_internal\_beartype.py:35: UserWarning: unhashable type: 'list'
  warnings.warn(f"{e}")
The torchaudio backend is switched to 'soundfile'. Note that 'sox_io' is not supported on Windows.


1
NVIDIA GeForce RTX 4070
cuda


c:\Users\avish\OneDrive\Desktop\thesis_research\avishai_work\env\lib\site-packages\speechbrain\utils\torch_audio_backend.py:22: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")


In [2]:
from ASV_utils.tdcf_functions import compute_t_dcf


import pickle
with open('objects_asv_and_cm_time_embeddings_eval_avg_score_both_asv_protocol.pkl', 'rb') as f:
    results_eval = pickle.load(f)

In [3]:
from ASV_utils.data_loading import *



models_folder = "Deep_Learning_Codes/best_models/fixed order/models/"

data_path_male = "Data/pmf_both/not_normalize/male/"

data_path_female = "Data/pmf_both/not_normalize/female/"

male_embedded_groups_1_1,male_embedded_groups_1_2,male_embedded_groups_1_3,male_chosen_labels_1_1_is_spoofed,male_chosen_labels_2_1_is_spoofed,male_chosen_labels_3_1_is_spoofed,male_chosen_labels_numeric_1_1,male_chosen_labels_numeric_2_1,male_chosen_labels_numeric_3_1,male_chosen_labels_1_1_attack_logical,male_chosen_labels_2_1_attack_logical,male_chosen_labels_3_1_attack_logical,male_chosen_labels_1_1_name,male_chosen_labels_2_1_name,male_chosen_labels_3_1_name,male_chosen_labels_1_1_speaker_id,male_chosen_labels_2_1_speaker_id,male_chosen_labels_3_1_speaker_id,male_chosen_labels_1_1_sex,male_chosen_labels_2_1_sex,male_chosen_labels_3_1_sex  = load_data_male(data_path_male)

female_embedded_groups_1_1,female_embedded_groups_1_2,female_embedded_groups_1_3,female_chosen_labels_1_1_is_spoofed,female_chosen_labels_2_1_is_spoofed,female_chosen_labels_3_1_is_spoofed,female_chosen_labels_numeric_1_1,female_chosen_labels_numeric_2_1,female_chosen_labels_numeric_3_1,female_chosen_labels_1_1_attack_logical,female_chosen_labels_2_1_attack_logical,female_chosen_labels_3_1_attack_logical,female_chosen_labels_1_1_name,female_chosen_labels_2_1_name,female_chosen_labels_3_1_name,female_chosen_labels_1_1_speaker_id,female_chosen_labels_2_1_speaker_id,female_chosen_labels_3_1_speaker_id,female_chosen_labels_1_1_sex,female_chosen_labels_2_1_sex,female_chosen_labels_3_1_sex  = load_data_female(data_path_female)


import utils.my_functions as my_functions

columns_names,max_name_length = my_functions.get_columns_names_feature_importance(substruct=True)
true_channels_indexes = np.array(my_functions.get_real_channel(np.linspace(start=1, stop=len(columns_names), num=len(columns_names)),len(columns_names)))
true_channels_indexes = true_channels_indexes - 1
true_channels_indexes = true_channels_indexes.astype(int)
columns_names = np.array(columns_names)

female_embedded_groups_1_1 = female_embedded_groups_1_1[:,true_channels_indexes]
female_embedded_groups_1_2 = female_embedded_groups_1_2[:,true_channels_indexes]
female_embedded_groups_1_3 = female_embedded_groups_1_3[:,true_channels_indexes]

male_embedded_groups_1_1 = male_embedded_groups_1_1[:,true_channels_indexes]
male_embedded_groups_1_2 = male_embedded_groups_1_2[:,true_channels_indexes]
male_embedded_groups_1_3 = male_embedded_groups_1_3[:,true_channels_indexes]

from sklearn.preprocessing import StandardScaler
import re

your_list = columns_names[true_channels_indexes]
index_mapping ={}

# Define the custom sorting order for distance metrics
distance_metrics = [
    'Chi-square',
    'Correlation',
    'Hellinger',
    'Intersection',
    'Jensen-Shannon',
    'Kullback-Leibler Divergence',
    'Modified Kolmogorov-Smirnov',
    'Symmetrised Kullback-Leibler'
]

def custom_sort_key(item):
    # Use regex to extract filter type, channel number, and distance metric
    match = re.search(r'filter-(gammatone|gammtone_inv)-channel-(\d+)-distance-(.+?)-\[d_', item)
    if match:
        filter_type = match.group(1)  # 'gammatone' or 'gammtone_inv'
        channel = int(match.group(2))
        distance_metric = match.group(3)
        
        # Prioritize 'gammatone' before 'gammtone_inv'
        filter_priority = 0 if filter_type == 'gammatone' else 1

        # Sort by filter type, then by distance metric, and finally by channel
        return (filter_priority, distance_metrics.index(distance_metric), channel)

    else:
        # If the regex doesn't match, push the item to the end
        return (999, 999, 999)

# Sort the list based on the custom order
sorted_list = sorted(your_list, key=custom_sort_key)

for new_index, item in enumerate(sorted_list):
    old_index = np.where(columns_names[true_channels_indexes] == item)[0][0]
    index_mapping[old_index] = new_index
    
male_embedded_groups_1_1 = male_embedded_groups_1_1[:,list(index_mapping.keys())]
male_embedded_groups_1_2 = male_embedded_groups_1_2[:,list(index_mapping.keys())]
male_embedded_groups_1_3 = male_embedded_groups_1_3[:,list(index_mapping.keys())]

female_embedded_groups_1_1 = female_embedded_groups_1_1[:,list(index_mapping.keys())]
female_embedded_groups_1_2 = female_embedded_groups_1_2[:,list(index_mapping.keys())]
female_embedded_groups_1_3 = female_embedded_groups_1_3[:,list(index_mapping.keys())]



In [4]:
# define the subchannel model network
import torch
import torch.nn as nn
from utils.AMSloss import AdMSoftmaxLoss
import pickle
# define the subchannel model network
import torch.nn as nn
class SubChannelNetwork(nn.Module):
    def __init__(self, input_channel_size, output_channel_size):
        super(SubChannelNetwork, self).__init__()
        self.input_layer = nn.Linear(input_channel_size, output_channel_size)
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(p=0.2)
        self.BN_4 = nn.BatchNorm1d(output_channel_size) 

        
    def forward(self, x):
        x = self.input_layer(x)
        x = self.BN_4(x) 
        x = self.sigmoid(x)
        x = self.dropout(x)
        return x

# define the model network
class DNN(nn.Module):
    def __init__(self, input_channel_size, num_subnetworks, output_channel_size, final_output_size):
        super(DNN, self).__init__()
        self.SubChannelNetwork = nn.ModuleList([
            SubChannelNetwork(input_channel_size, output_channel_size) for _ in range(num_subnetworks)
        ])
        self.fc_between_subnet = nn.Linear(num_subnetworks * output_channel_size,40)
        self.BN = nn.BatchNorm1d(40)
        self.fc = nn.Linear(40, final_output_size)
        self.sigmoid = nn.Sigmoid()
        self.droupout = nn.Dropout(p=0.2)
        self.loss = nn.BCEWithLogitsLoss()
        self.optimizer = None
        self.scheduler = None
        
        
    def forward(self, x):
        subnetwork_outputs = [self.SubChannelNetwork[idx](x[:, idx*input_channel_size:(idx+1)*input_channel_size].to(device)) for idx in range(len(self.SubChannelNetwork))]
        combined_output = torch.cat(subnetwork_outputs, dim=1)
        x = self.fc_between_subnet(combined_output)    
        x = self.BN(x)
        x = self.sigmoid(x)
        x = self.droupout(x)
        output = self.fc(x)
        return output 
    
    @staticmethod
    def count_parameters(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)



# Just for checking the model and see the number of parameters
num_SubChannelNetwork = 16
input_channel_size = 10
output_channel_size = 5
final_output_size = 1
model = []
model = DNN(input_channel_size, num_SubChannelNetwork, output_channel_size, final_output_size)
model = model.to(device)
print(model)
n = DNN.count_parameters(model)
print("Number of parameters: %s" % n)
model = model.to(device) # send the model to the device

model = pickle.load(open(models_folder + "Male_best_model_8_6.pkl", 'rb'))

model.eval()

spoof_model_male = copy.deepcopy(model)

DNN(
  (SubChannelNetwork): ModuleList(
    (0-15): 16 x SubChannelNetwork(
      (input_layer): Linear(in_features=10, out_features=5, bias=True)
      (sigmoid): Sigmoid()
      (dropout): Dropout(p=0.2, inplace=False)
      (BN_4): BatchNorm1d(5, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (fc_between_subnet): Linear(in_features=80, out_features=40, bias=True)
  (BN): BatchNorm1d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc): Linear(in_features=40, out_features=1, bias=True)
  (sigmoid): Sigmoid()
  (droupout): Dropout(p=0.2, inplace=False)
  (loss): BCEWithLogitsLoss()
)
Number of parameters: 4401


In [5]:
import torch
import torch.nn as nn
from utils.AMSloss import AdMSoftmaxLoss
from utils.OCSoftmaxloss import OCSoftmax
import torch.nn as nn

class SubChannelNetwork(nn.Module):
    def __init__(self, input_channel_size, output_channel_size):
        super(SubChannelNetwork, self).__init__()
        self.input_layer = nn.Linear(input_channel_size, output_channel_size)
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(p=0.2)
        self.BN_4 = nn.BatchNorm1d(output_channel_size) 
   
    def forward(self, x):
        res = x
        x = self.input_layer(x)
        x = self.BN_4(x) 
        x = self.sigmoid(x)
        x = self.dropout(x)
        x = res + x 
        return x

class DNN(nn.Module):
    def __init__(self, input_channel_size, num_subnetworks, output_channel_size, final_output_size, r_real , r_fake,alpha):
        super(DNN, self).__init__()
        self.SubChannelNetwork = nn.ModuleList([
            SubChannelNetwork(input_channel_size, output_channel_size) for _ in range(num_subnetworks)
        ])
        self.fc_between_subnet = nn.Linear(num_subnetworks * output_channel_size,40)
        self.BN = nn.BatchNorm1d(40)
        self.fc = nn.Linear(40, final_output_size)
        self.sigmoid = nn.Sigmoid()
        self.droupout = nn.Dropout(p=0.2)
        self.r_real = r_real
        self.r_fake = r_fake
        self.alpha = alpha
        self.loss = OCSoftmax(feat_dim = final_output_size, r_real = self.r_real, r_fake = self.r_fake, alpha = self.alpha)
        self.optimizer = None
        self.scheduler = None
        
        
    def forward(self, x):
        subnetwork_outputs = [self.SubChannelNetwork[idx](x[:, idx*input_channel_size:(idx+1)*input_channel_size].to(device)) for idx in range(len(self.SubChannelNetwork))]
        combined_output = torch.cat(subnetwork_outputs, dim=1)
        x = self.fc_between_subnet(combined_output)    
        x = self.BN(x)
        x = self.sigmoid(x)
        x = self.droupout(x)
        output = self.fc(x)
        return output 
    
    @staticmethod
    def count_parameters(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)

r_real = 0.5 
r_fake = 0.1
alpha = 20
# Example usage
num_SubChannelNetwork = 16
input_channel_size = 10
output_channel_size = 10
final_output_size = 16*3
model = DNN(input_channel_size, num_SubChannelNetwork, output_channel_size, final_output_size,r_real = r_real , r_fake = r_fake,alpha = alpha)


model = model.to(device) # send the model to the device

model = pickle.load(open(models_folder + "Female_best_model_10_12.pkl", 'rb'))

model.eval()

spoof_model_female = copy.deepcopy(model)


In [6]:
scaler_male = StandardScaler(with_mean = True, with_std = True)
scaler_male.fit(male_embedded_groups_1_1)
mean_features = scaler_male.mean_
std_features = scaler_male.scale_
male_embedded_groups_1_1_norm = scaler_male.transform(male_embedded_groups_1_1)
male_embedded_groups_1_2_norm = scaler_male.transform(male_embedded_groups_1_2)
male_embedded_groups_1_3_norm = scaler_male.transform(male_embedded_groups_1_3)


scaler_female = StandardScaler(with_mean = True, with_std = True)
scaler_female.fit(female_embedded_groups_1_1)
mean_features = scaler_female.mean_
std_features = scaler_female.scale_
female_embedded_groups_1_1_norm = scaler_female.transform(female_embedded_groups_1_1)
female_embedded_groups_1_2_norm = scaler_female.transform(female_embedded_groups_1_2)
female_embedded_groups_1_3_norm = scaler_female.transform(female_embedded_groups_1_3)

In [7]:
embedded_groups_1_1_norm,embedded_groups_1_2_norm,embedded_groups_1_3_norm,chosen_labels_1_1_is_spoofed,chosen_labels_2_1_is_spoofed,chosen_labels_3_1_is_spoofed,chosen_labels_numeric_1_1,chosen_labels_numeric_2_1,chosen_labels_numeric_3_1,chosen_labels_1_1_attack_logical,chosen_labels_2_1_attack_logical,chosen_labels_3_1_attack_logical,chosen_labels_1_1_name,chosen_labels_2_1_name,chosen_labels_3_1_name,chosen_labels_1_1_speaker_id,chosen_labels_2_1_speaker_id,chosen_labels_3_1_speaker_id,chosen_labels_1_1_sex,chosen_labels_2_1_sex,chosen_labels_3_1_sex    =  concatenate_data(male_embedded_groups_1_1_norm,male_embedded_groups_1_2_norm,male_embedded_groups_1_3_norm,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    male_chosen_labels_1_1_is_spoofed,male_chosen_labels_2_1_is_spoofed,male_chosen_labels_3_1_is_spoofed,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    male_chosen_labels_numeric_1_1,male_chosen_labels_numeric_2_1,male_chosen_labels_numeric_3_1,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    male_chosen_labels_1_1_attack_logical,male_chosen_labels_2_1_attack_logical,male_chosen_labels_3_1_attack_logical,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    male_chosen_labels_1_1_name,male_chosen_labels_2_1_name,male_chosen_labels_3_1_name,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    male_chosen_labels_1_1_speaker_id,male_chosen_labels_2_1_speaker_id, male_chosen_labels_3_1_speaker_id,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    male_chosen_labels_1_1_sex,male_chosen_labels_2_1_sex,male_chosen_labels_3_1_sex,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    female_embedded_groups_1_1_norm,female_embedded_groups_1_2_norm,female_embedded_groups_1_3_norm,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    female_chosen_labels_1_1_is_spoofed,female_chosen_labels_2_1_is_spoofed,female_chosen_labels_3_1_is_spoofed,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    female_chosen_labels_numeric_1_1,female_chosen_labels_numeric_2_1,female_chosen_labels_numeric_3_1,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    female_chosen_labels_1_1_attack_logical,female_chosen_labels_2_1_attack_logical,female_chosen_labels_3_1_attack_logical,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    female_chosen_labels_1_1_name,female_chosen_labels_2_1_name,female_chosen_labels_3_1_name,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    female_chosen_labels_1_1_speaker_id,female_chosen_labels_2_1_speaker_id,female_chosen_labels_3_1_speaker_id,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    female_chosen_labels_1_1_sex,female_chosen_labels_2_1_sex,female_chosen_labels_3_1_sex)


In [8]:
embedded_groups_1_1,embedded_groups_1_2,embedded_groups_1_3,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_    =  concatenate_data(male_embedded_groups_1_1,male_embedded_groups_1_2,male_embedded_groups_1_3,
                                                                                                                        male_chosen_labels_1_1_is_spoofed,male_chosen_labels_2_1_is_spoofed,male_chosen_labels_3_1_is_spoofed,
                                                                                                                        male_chosen_labels_numeric_1_1,male_chosen_labels_numeric_2_1,male_chosen_labels_numeric_3_1,
                                                                                                                        male_chosen_labels_1_1_attack_logical,male_chosen_labels_2_1_attack_logical,male_chosen_labels_3_1_attack_logical,
                                                                                                                        male_chosen_labels_1_1_name,male_chosen_labels_2_1_name,male_chosen_labels_3_1_name,
                                                                                                                        male_chosen_labels_1_1_speaker_id,male_chosen_labels_2_1_speaker_id, male_chosen_labels_3_1_speaker_id,
                                                                                                                        male_chosen_labels_1_1_sex,male_chosen_labels_2_1_sex,male_chosen_labels_3_1_sex,
                                                                                                                        
                                                                                                                        female_embedded_groups_1_1,female_embedded_groups_1_2,female_embedded_groups_1_3,
                                                                                                                        female_chosen_labels_1_1_is_spoofed,female_chosen_labels_2_1_is_spoofed,female_chosen_labels_3_1_is_spoofed,
                                                                                                                        female_chosen_labels_numeric_1_1,female_chosen_labels_numeric_2_1,female_chosen_labels_numeric_3_1,
                                                                                                                        female_chosen_labels_1_1_attack_logical,female_chosen_labels_2_1_attack_logical,female_chosen_labels_3_1_attack_logical,
                                                                                                                        female_chosen_labels_1_1_name,female_chosen_labels_2_1_name,female_chosen_labels_3_1_name,
                                                                                                                        female_chosen_labels_1_1_speaker_id,female_chosen_labels_2_1_speaker_id,female_chosen_labels_3_1_speaker_id,
                                                                                                                        female_chosen_labels_1_1_sex,female_chosen_labels_2_1_sex,female_chosen_labels_3_1_sex)
scaler_all = StandardScaler(with_mean = True, with_std = True)
scaler_all.fit(embedded_groups_1_1)
mean_features = scaler_female.mean_
std_features = scaler_female.scale_
embedded_groups_1_1_all = scaler_all.transform(embedded_groups_1_1)
embedded_groups_1_2_all = scaler_all.transform(embedded_groups_1_2)
embedded_groups_1_3_all = scaler_all.transform(embedded_groups_1_3)

In [9]:
train_dataset_all = dataset(data = embedded_groups_1_1_norm , is_spoofed = chosen_labels_1_1_is_spoofed , chosen_labels_numeric = chosen_labels_numeric_1_1,
                        attack_logical = chosen_labels_1_1_attack_logical, name = chosen_labels_1_1_name , speaker_id = chosen_labels_1_1_speaker_id, sex = chosen_labels_1_1_sex ,data_transform = None , labels_transform = None,
                        data_for_gender_classification = None,data_without_separation = embedded_groups_1_1_all);

Dev_dataset_all = dataset(data = embedded_groups_1_2_norm , is_spoofed = chosen_labels_2_1_is_spoofed , chosen_labels_numeric = chosen_labels_numeric_2_1,
                        attack_logical = chosen_labels_2_1_attack_logical, name = chosen_labels_2_1_name , speaker_id = chosen_labels_2_1_speaker_id, sex = chosen_labels_2_1_sex ,  data_transform = None , labels_transform = None,
                        data_for_gender_classification = None,data_without_separation = embedded_groups_1_2_all);


Eval_dataset_all = dataset(data = embedded_groups_1_3_norm , is_spoofed = chosen_labels_3_1_is_spoofed , chosen_labels_numeric = chosen_labels_numeric_3_1,
                        attack_logical = chosen_labels_3_1_attack_logical, name = chosen_labels_3_1_name , speaker_id = chosen_labels_3_1_speaker_id, sex = chosen_labels_3_1_sex , data_transform = None , labels_transform = None,
                        data_for_gender_classification = None,data_without_separation = embedded_groups_1_3_all);


In [10]:
from ASV_utils.data_loading import *


models_folder = "ECAPA_TDNN/inference_models/models_both_not_normalize/"

data_path_male = "Data/male_vs_female_DB_models/16_bits/none/male/"

data_path_female = "Data/male_vs_female_DB_models/16_bits/none/female/"

g_male_embedded_groups_1_1,g_male_embedded_groups_1_2,g_male_embedded_groups_1_3,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_  = load_data_male(data_path_male)

g_female_embedded_groups_1_1,g_female_embedded_groups_1_2,g_female_embedded_groups_1_3,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_  = load_data_female(data_path_female)

embedded_groups_1_1,embedded_groups_1_2,embedded_groups_1_3,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_    =  concatenate_data(g_male_embedded_groups_1_1,g_male_embedded_groups_1_2,g_male_embedded_groups_1_3,
                                                                                                                    male_chosen_labels_1_1_is_spoofed,male_chosen_labels_2_1_is_spoofed,male_chosen_labels_3_1_is_spoofed,
                                                                                                                    male_chosen_labels_numeric_1_1,male_chosen_labels_numeric_2_1,male_chosen_labels_numeric_3_1,
                                                                                                                    male_chosen_labels_1_1_attack_logical,male_chosen_labels_2_1_attack_logical,male_chosen_labels_3_1_attack_logical,
                                                                                                                    male_chosen_labels_1_1_name,male_chosen_labels_2_1_name,male_chosen_labels_3_1_name,
                                                                                                                    male_chosen_labels_1_1_speaker_id,male_chosen_labels_2_1_speaker_id, male_chosen_labels_3_1_speaker_id,
                                                                                                                    male_chosen_labels_1_1_sex,male_chosen_labels_2_1_sex,male_chosen_labels_3_1_sex,
                                                                                                                    
                                                                                                                    g_female_embedded_groups_1_1,g_female_embedded_groups_1_2,g_female_embedded_groups_1_3,
                                                                                                                    female_chosen_labels_1_1_is_spoofed,female_chosen_labels_2_1_is_spoofed,female_chosen_labels_3_1_is_spoofed,
                                                                                                                    female_chosen_labels_numeric_1_1,female_chosen_labels_numeric_2_1,female_chosen_labels_numeric_3_1,
                                                                                                                    female_chosen_labels_1_1_attack_logical,female_chosen_labels_2_1_attack_logical,female_chosen_labels_3_1_attack_logical,
                                                                                                                    female_chosen_labels_1_1_name,female_chosen_labels_2_1_name,female_chosen_labels_3_1_name,
                                                                                                                    female_chosen_labels_1_1_speaker_id,female_chosen_labels_2_1_speaker_id,female_chosen_labels_3_1_speaker_id,
                                                                                                                    female_chosen_labels_1_1_sex,female_chosen_labels_2_1_sex,female_chosen_labels_3_1_sex)
chosen_labels_1_1_sex = np.array([elem[0] for elem in chosen_labels_1_1_sex])
                                 
chosen_labels_2_1_sex = np.array([elem[0] for elem in chosen_labels_2_1_sex])

chosen_labels_3_1_sex = np.array([elem[0] for elem in chosen_labels_3_1_sex])


In [11]:



import utils.my_functions as my_functions

columns_names,max_name_length = my_functions.get_columns_names_feature_importance(substruct=True)
true_channels_indexes = np.array(my_functions.get_real_channel(np.linspace(start=1, stop=len(columns_names), num=len(columns_names)),len(columns_names)))
true_channels_indexes = true_channels_indexes - 1
true_channels_indexes = true_channels_indexes.astype(int)
columns_names = np.array(columns_names)
'''
g_female_embedded_groups_1_1 = g_female_embedded_groups_1_1[:,true_channels_indexes]
g_female_embedded_groups_1_2 = g_female_embedded_groups_1_2[:,true_channels_indexes]
g_female_embedded_groups_1_3 = g_female_embedded_groups_1_3[:,true_channels_indexes]

g_male_embedded_groups_1_1 = g_male_embedded_groups_1_1[:,true_channels_indexes]
g_male_embedded_groups_1_2 = g_male_embedded_groups_1_2[:,true_channels_indexes]
g_male_embedded_groups_1_3 = g_male_embedded_groups_1_3[:,true_channels_indexes]
'''

embedded_groups_1_1 = embedded_groups_1_1[:,true_channels_indexes]
embedded_groups_1_2 = embedded_groups_1_2[:,true_channels_indexes]
embedded_groups_1_3 = embedded_groups_1_3[:,true_channels_indexes]

from sklearn.preprocessing import StandardScaler
import re

your_list = columns_names[true_channels_indexes]
index_mapping ={}

# Define the custom sorting order for distance metrics
distance_metrics = [
    'Chi-square',
    'Correlation',
    'Hellinger',
    'Intersection',
    'Jensen-Shannon',
    'Kullback-Leibler Divergence',
    'Modified Kolmogorov-Smirnov',
    'Symmetrised Kullback-Leibler'
]

def custom_sort_key(item):
    # Use regex to extract filter type, channel number, and distance metric
    match = re.search(r'filter-(gammatone|gammtone_inv)-channel-(\d+)-distance-(.+?)-\[d_', item)
    if match:
        filter_type = match.group(1)  # 'gammatone' or 'gammtone_inv'
        channel = int(match.group(2))
        distance_metric = match.group(3)
        
        # Prioritize 'gammatone' before 'gammtone_inv'
        filter_priority = 0 if filter_type == 'gammatone' else 1

        # Sort by filter type, then by distance metric, and finally by channel
        return (filter_priority, distance_metrics.index(distance_metric), channel)

    else:
        # If the regex doesn't match, push the item to the end
        return (999, 999, 999)

# Sort the list based on the custom order
sorted_list = sorted(your_list, key=custom_sort_key)

for new_index, item in enumerate(sorted_list):
    old_index = np.where(columns_names[true_channels_indexes] == item)[0][0]
    index_mapping[old_index] = new_index
    

embedded_groups_1_1 = embedded_groups_1_1[:,list(index_mapping.keys())]
embedded_groups_1_2 = embedded_groups_1_2[:,list(index_mapping.keys())]
embedded_groups_1_3 = embedded_groups_1_3[:,list(index_mapping.keys())]




In [12]:
scaler = StandardScaler(with_mean = True, with_std = True)
scaler.fit(embedded_groups_1_1)
mean_features = scaler.mean_
std_features = scaler.scale_

embedded_groups_1_1 = scaler.transform(embedded_groups_1_1)
embedded_groups_1_2 = scaler.transform(embedded_groups_1_2)
embedded_groups_1_3 = scaler.transform(embedded_groups_1_3)

train_dataset_all.data_for_gender_classification = embedded_groups_1_1
Dev_dataset_all.data_for_gender_classification   = embedded_groups_1_2
Eval_dataset_all.data_for_gender_classification  = embedded_groups_1_3


train_dataset_all.sex = pd.Series([elem[0] for elem in train_dataset_all.sex])
Dev_dataset_all.sex = pd.Series([elem[0] for elem in Dev_dataset_all.sex])
Eval_dataset_all.sex = pd.Series([elem[0] for elem in Eval_dataset_all.sex])

In [13]:
labels_eval_female = torch.Tensor(Eval_dataset_all.is_spoofed[(Eval_dataset_all.sex == 'female').values].values).cpu().numpy().copy()
total_data = torch.Tensor(Eval_dataset_all.data[(Eval_dataset_all.sex == 'female').values]).cpu().numpy().copy()
with torch.no_grad():
    model = model.to(device)
    output = spoof_model_female(torch.Tensor(total_data).to(device))
    _ , score_eval_female = spoof_model_female.loss(torch.Tensor(output).to(device),None)
    score_eval_female = -1*score_eval_female
    
score_eval_female = score_eval_female.cpu().numpy().copy()

eer_eval_female, thr = my_functions.compute_eer(labels_eval_female,score_eval_female) # compute equal error rate

print(f"\tEval Female labels gender EER: ({eer_eval_female}%), {thr}")


	Eval Female labels gender EER: (0.10129787907565715%), -0.8330833536418566


In [14]:
labels_eval_male = torch.Tensor(Eval_dataset_all.is_spoofed[(Eval_dataset_all.sex == 'male').values].values).cpu().numpy().copy()
total_data = torch.Tensor(Eval_dataset_all.data[(Eval_dataset_all.sex == 'male').values]).cpu().numpy().copy()
with torch.no_grad():
    model = model.to(device)
    score_eval_male = torch.sigmoid(spoof_model_male(torch.Tensor(total_data).to(device)))
    
score_eval_male = score_eval_male.cpu().numpy().copy()

eer_eval_male, thr = my_functions.compute_eer(labels_eval_male,score_eval_male) # compute equal error rate

print(f"\t Eval Male EER: ({100*eer_eval_male}%) , {thr}")

	 Eval Male EER: (8.672798948696906%) , 0.04048307612539639


# implementation of t-DCF in 2 subsystems

In [15]:
bonafide_score_cm_female_eval = score_eval_female[labels_eval_female == 0]
bonafide_score_cm_female_eval = -1*bonafide_score_cm_female_eval

spoof_score_cm_female_eval = score_eval_female[labels_eval_female == 1]
spoof_score_cm_female_eval  = -1*spoof_score_cm_female_eval

Prior_spoof = 0.05

target_scores_female_eval = results_eval.loc[(results_eval['asv_label'] ==  "target") & (results_eval['its_male_ground_truth'] == 'female') ]["asv_score_female"].values
nontarget_scores_female_eval = results_eval.loc[(results_eval['asv_label'] == "nontarget") & (results_eval['its_male_ground_truth'] == 'female') ]["asv_score_female"].values
spoof_scores_female_eval = results_eval.loc[(results_eval['asv_label'] ==  "spoof") & (results_eval['its_male_ground_truth'] == 'female')]["asv_score_female"].values

list_asv_score_female = list_asv_score = [0.401]
type = 'constrained'
_ ,_,_  = compute_t_dcf(bonafide_score_cm_female_eval,spoof_score_cm_female_eval,Prior_spoof,target_scores_female_eval,nontarget_scores_female_eval,spoof_scores_female_eval,list_asv_score,type)

The t-DCF evaluation type is: constrained
t-DCF evaluation from [Nbona=5072, Nspoof=44226] trials

t-DCF MODEL
   Ptar         =  0.94050 (Prior probability of target user)
   Pnon         =  0.00950 (Prior probability of nontarget user)
   Pspoof       =  0.05000 (Prior probability of spoofing attack)
   Cfa_asv      = 10.00000 (Cost of ASV falsely accepting a nontarget)
   Cmiss_asv    =  1.00000 (Cost of ASV falsely rejecting target speaker)
   Cfa_cm       = 10.00000 (Cost of CM falsely passing a spoof to ASV system)
   Cmiss_cm     =  1.00000 (Cost of CM falsely blocking target utterance which never reaches ASV)

   Implied normalized t-DCF function (depends on t-DCF parameters and ASV errors), s=CM threshold)
   tDCF_norm(s) =  3.00467 x Pmiss_cm(s) + Pfa_cm(s)

the asv threshold is from eer on asv development set  0.401
the CM thresholds is:  [-0.82588501 -0.82488501 -0.82439852 ...  0.94250655  0.94292605
  0.95455754]
the CM threshold min is: 0.7508225440979004
the tDCF_norm i

In [16]:

bonafide_score_cm_male_eval = 1-score_eval_male[labels_eval_male == 0].flatten()
spoof_score_cm_male_eval = 1-score_eval_male[labels_eval_male == 1].flatten()

Prior_spoof = 0.05

target_scores_male_eval = results_eval.loc[(results_eval['asv_label'] ==  "target") & (results_eval['its_male_ground_truth'] == 'male') ]["asv_score_male"].values
nontarget_scores_male_eval = results_eval.loc[(results_eval['asv_label'] == "nontarget") & (results_eval['its_male_ground_truth'] == 'male') ]["asv_score_male"].values
spoof_scores_male_eval = results_eval.loc[(results_eval['asv_label'] ==  "spoof") & (results_eval['its_male_ground_truth'] == 'male')]["asv_score_male"].values

list_asv_score_male = list_asv_score = [0.352]
type = 'constrained'
_ ,_,_  = compute_t_dcf(bonafide_score_cm_male_eval,spoof_score_cm_male_eval,Prior_spoof,target_scores_male_eval,nontarget_scores_male_eval,spoof_scores_male_eval,list_asv_score,type)

The t-DCF evaluation type is: constrained
t-DCF evaluation from [Nbona=2283, Nspoof=19656] trials

t-DCF MODEL
   Ptar         =  0.94050 (Prior probability of target user)
   Pnon         =  0.00950 (Prior probability of nontarget user)
   Pspoof       =  0.05000 (Prior probability of spoofing attack)
   Cfa_asv      = 10.00000 (Cost of ASV falsely accepting a nontarget)
   Cmiss_asv    =  1.00000 (Cost of ASV falsely rejecting target speaker)
   Cfa_cm       = 10.00000 (Cost of CM falsely passing a spoof to ASV system)
   Cmiss_cm     =  1.00000 (Cost of CM falsely blocking target utterance which never reaches ASV)

   Implied normalized t-DCF function (depends on t-DCF parameters and ASV errors), s=CM threshold)
   tDCF_norm(s) =  2.72851 x Pmiss_cm(s) + Pfa_cm(s)

the asv threshold is from eer on asv development set  0.352
the CM thresholds is:  [4.97268677e-04 1.49726868e-03 1.50895119e-03 ... 9.96871948e-01
 9.96981561e-01 9.97154653e-01]
the CM threshold min is: 0.90961152315139

In [17]:
def obtain_asv_error_rates_male_female(tar_asv_male, non_asv_male, spoof_asv_male, asv_threshold_male,tar_asv_female, non_asv_female, spoof_asv_female, asv_threshold_female):
    
    # False alarm and miss rates for ASV
    Pfa_asv_total = np.sum(non_asv_male >= asv_threshold_male) / (non_asv_male.size+non_asv_female.size) + np.sum(non_asv_female >= asv_threshold_female) / (non_asv_male.size+non_asv_female.size)
    Pmiss_asv_total = np.sum(tar_asv_male < asv_threshold_male) / (tar_asv_male.size+tar_asv_female.size) + np.sum(tar_asv_female < asv_threshold_female) / (tar_asv_male.size+tar_asv_female.size)

    # Rate of rejecting spoofs in ASV
    if spoof_asv_male.size+spoof_asv_female.size == 0:
        Pmiss_spoof_asv_total = None
        Pfa_spoof_asv_total = None
    else:
        Pmiss_spoof_asv_total = np.sum(spoof_asv_male < asv_threshold_male) / (spoof_asv_male.size + spoof_asv_female.size) + np.sum(spoof_asv_female < asv_threshold_female) / (spoof_asv_male.size + spoof_asv_female.size)
        
        # False Acceptance Rate for Spoofing Attacks
        Pfa_spoof_asv_total = np.sum(spoof_asv_male >= asv_threshold_male) / (spoof_asv_male.size + spoof_asv_female.size) + np.sum(spoof_asv_female >= asv_threshold_female) / (spoof_asv_male.size + spoof_asv_female.size)

    return Pfa_asv_total, Pmiss_asv_total, Pmiss_spoof_asv_total, Pfa_spoof_asv_total

In [18]:
from tqdm import tqdm
def calc_p_cm_miss_fa(bonafide_score_cm_male,spoof_score_cm_male,bonafide_score_cm_female,spoof_score_cm_female,thresholds_male,thresholds_female):
    bonafide_male_size = bonafide_score_cm_male.size
    bonafide_female_size = bonafide_score_cm_female.size
    spoof_male_size = spoof_score_cm_male.size
    spoof_female_size = spoof_score_cm_female.size

    P_cm_miss_total = np.zeros((len(thresholds_male), len(thresholds_female)),dtype=np.float32)
    P_cm_fa_total = np.zeros((len(thresholds_male), len(thresholds_female)),dtype=np.float32)

    for i, thr_male in enumerate(thresholds_male):
        for j, thr_female in enumerate(thresholds_female):
            cm_miss_male = np.sum(bonafide_score_cm_male <= thr_male)
            cm_miss_female = np.sum(bonafide_score_cm_female <= thr_female)
            P_cm_miss_total[i, j] = (cm_miss_male + cm_miss_female) / (bonafide_male_size + bonafide_female_size)

            cm_fa_male = np.sum(spoof_score_cm_male > thr_male)
            cm_fa_female = np.sum(spoof_score_cm_female > thr_female)
            P_cm_fa_total[i, j] = (cm_fa_male + cm_fa_female) / (spoof_male_size + spoof_female_size)

    return P_cm_miss_total,P_cm_fa_total 



In [19]:
def compute_det_curve(target_scores, nontarget_scores):
    
    n_scores = target_scores.size + nontarget_scores.size
    all_scores = np.concatenate((target_scores, nontarget_scores))
    labels = np.concatenate((np.ones(target_scores.size), np.zeros(nontarget_scores.size)))

    # Sort labels based on scores
    indices = np.argsort(all_scores, kind='mergesort')
    labels = labels[indices]

    # Compute false rejection and false acceptance rates
    tar_trial_sums = np.cumsum(labels)
    nontarget_trial_sums = nontarget_scores.size - (np.arange(1, n_scores + 1) - tar_trial_sums)

    frr = np.concatenate((np.atleast_1d(0), tar_trial_sums / target_scores.size))  # false rejection rates
    far = np.concatenate((np.atleast_1d(1), nontarget_trial_sums / nontarget_scores.size))  # false acceptance rates
    thresholds = np.concatenate((np.atleast_1d(all_scores[indices[0]] - 0.001), all_scores[indices]))  # Thresholds are the sorted scores

    return frr, far, thresholds

In [20]:
from tqdm import tqdm
#using only one CM threshold
import multiprocessing

print_cost = True
def calc_t_dcf_spesific(target_scores_male,nontarget_scores_male,spoof_scores_male,spoof_score_cm_male,bonafide_score_cm_male,asv_score_male,thr_male,
                        target_scores_female,nontarget_scores_female,spoof_scores_female,spoof_score_cm_female,bonafide_score_cm_female,asv_score_female,thr_female,type_overlap = 'spesific'):
    cost_model = {
            'Ptar':(1-Prior_spoof)*0.99,         # Prior probability of target speaker
            'Pnon':(1-Prior_spoof)*0.01,         # Prior probability of nontarget speaker (zero-effort impostor)
            'Pspoof': Prior_spoof,        # Prior probability of spoofing attack
            'Cmiss_asv': 1,      # Cost of ASV falsely rejecting target
            'Cfa_asv': 10,       # Cost of ASV falsely accepting nontarget
            'Cmiss_cm': 1,       # Cost of CM falsely rejecting target
            'Cfa_cm': 10,          # Cost of CM falsely accepting spoof
            'Cmiss': 1,      
            'Cfa': 10,
            'Cfa_spoof': 10
        }

    if type_overlap != 'spesific' and type_overlap != 'all':
        raise ValueError('type_overlap should be either \"spesific\" or \"all\"')

    Pfa_asv, Pmiss_asv, Pmiss_spoof_asv, Pfa_spoof_asv = obtain_asv_error_rates_male_female(target_scores_male,nontarget_scores_male,spoof_scores_male,asv_score_male,target_scores_female,nontarget_scores_female,spoof_scores_female,asv_score_female)


    bonafide_score_cm = np.concatenate((bonafide_score_cm_male,bonafide_score_cm_female))

    spoof_score_cm = np.concatenate((spoof_score_cm_male,spoof_score_cm_female))

    # Obtain miss and false alarm rates of CM
    
    if type_overlap == 'all':
    
        _,_, CM_thresholds_male = compute_det_curve(bonafide_score_cm_male, spoof_score_cm_male)
        
        _,_, CM_thresholds_female = compute_det_curve(bonafide_score_cm_female, spoof_score_cm_female)

        P_cm_miss_total,P_cm_fa_total  = calc_p_cm_miss_fa(bonafide_score_cm_male,spoof_score_cm_male,bonafide_score_cm_female,spoof_score_cm_female,CM_thresholds_male,CM_thresholds_female)
    
    elif type_overlap == 'spesific':
        
        P_cm_miss_total,P_cm_fa_total  = calc_p_cm_miss_fa(bonafide_score_cm_male,spoof_score_cm_male,bonafide_score_cm_female,spoof_score_cm_female,thr_male,thr_female)



    # Constants - see ASVspoof 2019 evaluation plan
    # constrained
    
    C1 = cost_model['Ptar'] * (cost_model['Cmiss_cm'] - cost_model['Cmiss_asv'] * Pmiss_asv) - \
            cost_model['Pnon'] * cost_model['Cfa_asv'] * Pfa_asv
    C2 = cost_model['Cfa_cm'] * cost_model['Pspoof'] * (1 - Pmiss_spoof_asv)
    

    # Sanity check of the weights
    if C1 < 0 or C2 < 0:
        sys.exit('You should never see this error but I cannot evalute tDCF with negative weights - please check whether your ASV error rates are correctly computed?')

    # Obtain t-DCF curve for all thresholds


    A = C1 * P_cm_miss_total 


    B = C2 * P_cm_fa_total

    tDCF = A + B

    tDCF_norm = tDCF / np.minimum(C1, C2)
    
   
    
    '''
    #ver2 constrained
    C0 = cost_model['Ptar']*cost_model['Cmiss']*Pmiss_asv + cost_model['Pnon']*cost_model['Cfa']*Pfa_asv
    C1 = cost_model['Ptar'] * (cost_model['Cmiss'] - cost_model['Cmiss'] * Pmiss_asv) - \
         cost_model['Pnon'] * cost_model['Cfa'] * Pfa_asv
    C2 = cost_model['Cfa_spoof'] * cost_model['Pspoof'] * Pfa_spoof_asv
    # Obtain t-DCF curve for all thresholds
    tDCF = C0 +C1*P_cm_miss_total +C2*P_cm_fa_total

    # Normalized t-DCF

    tDCF_norm = tDCF / (C0 + np.minimum(C1, C2))
    '''
    '''
    P_a = (1-P_cm_miss_total) * Pmiss_asv
    P_b = (1-P_cm_miss_total) * Pfa_asv
    P_c = P_cm_fa_total*Pfa_spoof_asv
    P_d = P_cm_miss_total
    
    # Obtain t-DCF curve for all thresholds
    tDCF = cost_model['Cmiss']*cost_model['Ptar']*(P_a +P_d) + cost_model['Cfa']*cost_model['Pnon']*P_b + cost_model['Cfa_spoof']*cost_model['Pspoof']*P_c

    # Normalized t-DCF
    min1 = cost_model['Cfa']*cost_model['Pnon'] + cost_model['Cfa_spoof']*cost_model['Pspoof']
    min2 = cost_model['Cmiss']*cost_model['Ptar']
    
    tDCF_norm = tDCF / np.minimum(min1, min2)
    # Normalized t-DCF
    '''
    # Everything should be fine if reaching here.
    print_cost = False
    if print_cost:

        print('t-DCF evaluation from [Nbona={}, Nspoof={}] trials\n'.format(bonafide_score_cm.size, spoof_score_cm.size))
        print('t-DCF MODEL')
        print('   Ptar         = {:8.5f} (Prior probability of target user)'.format(cost_model['Ptar']))
        print('   Pnon         = {:8.5f} (Prior probability of nontarget user)'.format(cost_model['Pnon']))
        print('   Pspoof       = {:8.5f} (Prior probability of spoofing attack)'.format(cost_model['Pspoof']))
        print('   Cfa_asv      = {:8.5f} (Cost of ASV falsely accepting a nontarget)'.format(cost_model['Cfa_asv']))
        print('   Cmiss_asv    = {:8.5f} (Cost of ASV falsely rejecting target speaker)'.format(cost_model['Cmiss_asv']))
        print('   Cfa_cm       = {:8.5f} (Cost of CM falsely passing a spoof to ASV system)'.format(cost_model['Cfa_cm']))
        print('   Cmiss_cm     = {:8.5f} (Cost of CM falsely blocking target utterance which never reaches ASV)'.format(cost_model['Cmiss_cm']))
        print('\n   Implied normalized t-DCF function (depends on t-DCF parameters and ASV errors), s=CM threshold)')

        
    if type_overlap == 'spesific':     
        
        CM_thresholds_female = thr_female
        CM_thresholds_male = thr_male
        # print("tDCF_norm: ",tDCF_norm)
        # print("tDCF: ",tDCF)
        # print(f"P_cm_miss: {P_cm_miss_total}")
        # print(f"P_cm_fa: {P_cm_fa_total}")
        
        
    
    elif type_overlap == 'all':
        min_index = np.argmin(tDCF_norm)
        min_index_row, min_index_col = np.unravel_index(min_index, tDCF_norm.shape)    
        print(f"t-DCF norm min: {tDCF_norm[min_index_row, min_index_col]}")
        print(f"t-DCF min: {tDCF[min_index_row, min_index_col]}")
        print(f"P_cm_miss: {P_cm_miss[min_index_row,min_index_col]}")
        print(f"P_cm_fa: {P_cm_fa[min_index_row,min_index_col]}")
        print(f"CM threshold Male: {CM_thresholds_male[min_index_row]}")
        print(f"CM threshold Female: {CM_thresholds_female[min_index_col]}")
            
        
    return tDCF_norm, tDCF, P_cm_miss_total, P_cm_fa_total, CM_thresholds_male, CM_thresholds_female


# Eval with Labels:

In [21]:
import pickle

#asv male
target_scores_male = target_scores_male_eval 
nontarget_scores_male = nontarget_scores_male_eval
spoof_scores_male = spoof_scores_male_eval 
asv_score_male = 0.352
# cm male
bonafide_score_cm_male = bonafide_score_cm_male_eval
spoof_score_cm_male = spoof_score_cm_male_eval
thr_male =  0.9096115231513977 # from the t-dcf print output

#asv female
target_scores_female = target_scores_female_eval 
nontarget_scores_female = nontarget_scores_female_eval
spoof_scores_female = spoof_scores_female_eval 
asv_score_female = 0.401
#
bonafide_score_cm_female = bonafide_score_cm_female_eval
spoof_score_cm_female = spoof_score_cm_female_eval
thr_female =   0.7508225440979004 # from the t-dcf print output

_ ,_,_  = compute_t_dcf(bonafide_score_cm_male,spoof_score_cm_male,Prior_spoof,target_scores_male,nontarget_scores_male,spoof_scores_male,[asv_score_male],type)

_ ,_,_  = compute_t_dcf(bonafide_score_cm_female,spoof_score_cm_female,Prior_spoof,target_scores_female,nontarget_scores_female,spoof_scores_female,[asv_score_female],type)


The t-DCF evaluation type is: constrained
t-DCF evaluation from [Nbona=2283, Nspoof=19656] trials

t-DCF MODEL
   Ptar         =  0.94050 (Prior probability of target user)
   Pnon         =  0.00950 (Prior probability of nontarget user)
   Pspoof       =  0.05000 (Prior probability of spoofing attack)
   Cfa_asv      = 10.00000 (Cost of ASV falsely accepting a nontarget)
   Cmiss_asv    =  1.00000 (Cost of ASV falsely rejecting target speaker)
   Cfa_cm       = 10.00000 (Cost of CM falsely passing a spoof to ASV system)
   Cmiss_cm     =  1.00000 (Cost of CM falsely blocking target utterance which never reaches ASV)

   Implied normalized t-DCF function (depends on t-DCF parameters and ASV errors), s=CM threshold)
   tDCF_norm(s) =  2.72851 x Pmiss_cm(s) + Pfa_cm(s)

the asv threshold is from eer on asv development set  0.352
the CM thresholds is:  [4.97268677e-04 1.49726868e-03 1.50895119e-03 ... 9.96871948e-01
 9.96981561e-01 9.97154653e-01]
the CM threshold min is: 0.90961152315139

In [22]:
import numpy as np
import sklearn.metrics
from confidence_intervals import evaluate_with_conf_int
from confidence_intervals import get_bootstrap_indices, get_conf_int
from confidence_intervals.utils import barplot_with_ci
import warnings
warnings.filterwarnings("ignore")
# Percentage for the confidence interval
alpha = 5 

# Number of bootstrap samples to use (the run time will be proportional to this number). We set it to
# 50/alpha*100 to get enough samples in the tails.
num_bootstraps = int(50/alpha*100)

print(" Number of bootstraps: ", num_bootstraps)
print(" Alpha: ", alpha)

 Number of bootstraps:  1000
 Alpha:  5


In [23]:
def metric(labels, scores):
    print_cost = False
    Prior_spoof = 0.05
    
    asv_score = 0.397
    bonafide_score_cm_male =  scores[labels == 0]
    spoof_score_cm_male = scores[labels == 1]
    bonafide_score_cm_female = scores[labels == 2]
    spoof_score_cm_female = scores[labels == 3]
    cost_model = {
            'Ptar':(1-Prior_spoof)*0.99,         # Prior probability of target speaker
            'Pnon':(1-Prior_spoof)*0.01,         # Prior probability of nontarget speaker (zero-effort impostor)
            'Pspoof': Prior_spoof,        # Prior probability of spoofing attack
            'Cmiss_asv': 1,      # Cost of ASV falsely rejecting target
            'Cfa_asv': 10,       # Cost of ASV falsely accepting nontarget
            'Cmiss_cm': 1,       # Cost of CM falsely rejecting target
            'Cfa_cm': 10          # Cost of CM falsely accepting spoof
        }
    tDCF_norm, tDCF, P_cm_miss_total, P_cm_fa_total, thresholds_male, thresholds_female = calc_t_dcf_spesific(target_scores_male,nontarget_scores_male,spoof_scores_male,spoof_score_cm_male,bonafide_score_cm_male,asv_score_male,[thr_male],
                                                                                                                    target_scores_female,nontarget_scores_female,spoof_scores_female,spoof_score_cm_female,bonafide_score_cm_female,asv_score_female,[thr_female],type_overlap = 'spesific')

    return min(tDCF_norm)


#tDCF_norm, tDCF, P_cm_miss_total, P_cm_fa_total, thresholds_male, thresholds_female = calc_t_dcf_spesific(target_scores_male,nontarget_scores_male,spoof_scores_male,spoof_score_cm_male,bonafide_score_cm_male,asv_score_male,[thr_male],
#                                                                                                                    target_scores_female,nontarget_scores_female,spoof_scores_female,spoof_score_cm_female,bonafide_score_cm_female,asv_score_female,[thr_female],type_overlap = 'all')

speakers = Eval_dataset_all.speaker_id
scores = np.concatenate((bonafide_score_cm_male,spoof_score_cm_male,bonafide_score_cm_female,spoof_score_cm_female))
labels = np.concatenate((np.zeros(len(bonafide_score_cm_male)),np.ones(len(spoof_score_cm_male)),np.ones(len(bonafide_score_cm_female)) *2 ,np.ones(len(spoof_score_cm_female)) * 3))

res_eval_labels = evaluate_with_conf_int(samples = scores, metric = metric, labels = labels, conditions = speakers, num_bootstraps=num_bootstraps, alpha=alpha)

print(res_eval_labels)
#with open('results_total_tdcf_wth_labels.pkl', 'wb') as f:
#    pickle.dump(results_total_tdcf_wth_labels, f)

(array([0.2693365], dtype=float32), (0.24840762913227082, 0.2914756886661053))


# Eval with Preds:

In [24]:
import torchaudio
from speechbrain.pretrained import EncoderClassifier
gender_models = "ML_codes/pmf_both_classification_male_vs_female/new_female_and_male/female_vs_male_histograms/normalize_all_best_fixed_order/"
# load the model - the moodel that check if it's female or male
with open(os.path.join(gender_models,"gender_XGB_model_both_norm_male_vs_female_db_models_5_fixed_order.pkl"), 'rb') as fp:
#with open(os.path.join(models_folder,"gender_XGB_model_both_norm_male_vs_female_db_models.pkl"), 'rb') as fp:
    gender_model = pickle.load(fp)
    


In [25]:
labels_eval_male_gender_classification = torch.Tensor(Eval_dataset_all.is_spoofed[gender_model.predict(Eval_dataset_all.data_for_gender_classification) == 1]).cpu().numpy().copy()
total_data = torch.Tensor(Eval_dataset_all.data[gender_model.predict(Eval_dataset_all.data_for_gender_classification) == 1]).cpu().numpy().copy()
with torch.no_grad():
    model = model.to(device)
    score_eval_male_gender_classification = torch.sigmoid(spoof_model_male(torch.Tensor(total_data).to(device)))
    
score_eval_male_gender_classification = score_eval_male_gender_classification.cpu().numpy().copy()

eval_male_eer, _ = my_functions.compute_eer(labels_eval_male_gender_classification,score_eval_male_gender_classification) # compute equal error rate

print(f"\tEval male with gender classification EER: ({100*eval_male_eer}%) ")

	Eval male with gender classification EER: (8.522727272757948%) 


In [26]:
labels_eval_female_gender_classification = torch.Tensor(Eval_dataset_all.is_spoofed[gender_model.predict(Eval_dataset_all.data_for_gender_classification) == 0].values).cpu().numpy().copy()
total_data = torch.Tensor(Eval_dataset_all.data[gender_model.predict(Eval_dataset_all.data_for_gender_classification) == 0]).cpu().numpy().copy()
with torch.no_grad():
    model = model.to(device)
    output = spoof_model_female(torch.Tensor(total_data).to(device))
    _ , score_eval_female_gender_classification = spoof_model_female.loss(torch.Tensor(output).to(device),None)
    score_eval_female_gender_classification = -1*score_eval_female_gender_classification
    
score_eval_female_gender_classification = score_eval_female_gender_classification.cpu().numpy().copy()

eer, _ = my_functions.compute_eer(labels_eval_female_gender_classification,score_eval_female_gender_classification) # compute equal error rate

print(f"\tDev Female EER:({100*eer}%)")

	Dev Female EER:(10.227090078299229%)


In [27]:
bonafide_score_cm_female_eval = score_eval_female_gender_classification[labels_eval_female_gender_classification == 0]
bonafide_score_cm_female_eval = -1*bonafide_score_cm_female_eval

spoof_score_cm_female_eval = score_eval_female_gender_classification[labels_eval_female_gender_classification == 1]
spoof_score_cm_female_eval  = -1*spoof_score_cm_female_eval


target_scores_female_eval = results_eval.loc[(results_eval['asv_label'] ==  "target") & (results_eval['its_male'] == False) ]["asv_score_female"].values
nontarget_scores_female_eval = results_eval.loc[(results_eval['asv_label'] == "nontarget") & (results_eval['its_male'] == False) ]["asv_score_female"].values
spoof_scores_female_eval = results_eval.loc[(results_eval['asv_label'] ==  "spoof") & (results_eval['its_male'] == False)]["asv_score_female"].values
list_asv_score_female = list_asv_score = [0.401]


bonafide_score_cm_male_eval = score_eval_male_gender_classification[labels_eval_male_gender_classification == 0].flatten()
bonafide_score_cm_male_eval = 1-bonafide_score_cm_male_eval

spoof_score_cm_male_eval = score_eval_male_gender_classification[labels_eval_male_gender_classification == 1].flatten()
spoof_score_cm_male_eval = 1-spoof_score_cm_male_eval

Prior_spoof = 0.05
target_scores_male_eval = results_eval.loc[(results_eval['asv_label'] ==  "target") & (results_eval['its_male'] == True) ]["asv_score_male"].values
nontarget_scores_male_eval = results_eval.loc[(results_eval['asv_label'] == "nontarget") & (results_eval['its_male'] == True) ]["asv_score_male"].values
spoof_scores_male_eval = results_eval.loc[(results_eval['asv_label'] ==  "spoof") & (results_eval['its_male'] == True)]["asv_score_male"].values

list_asv_score_male = list_asv_score = [0.352]

In [28]:
#asv male
target_scores_male = target_scores_male_eval 
nontarget_scores_male = nontarget_scores_male_eval
spoof_scores_male = spoof_scores_male_eval 
asv_score_male = list_asv_score_male[0]
# cm male
bonafide_score_cm_male = bonafide_score_cm_male_eval
spoof_score_cm_male = spoof_score_cm_male_eval
thr_male =  0.9096115231513977 # from the t-dcf print output

#asv female
target_scores_female = target_scores_female_eval 
nontarget_scores_female = nontarget_scores_female_eval
spoof_scores_female = spoof_scores_female_eval 
asv_score_female = list_asv_score_female[0]
#
bonafide_score_cm_female = bonafide_score_cm_female_eval
spoof_score_cm_female = spoof_score_cm_female_eval
thr_female =   0.7508225440979004

_ ,_,_  = compute_t_dcf(bonafide_score_cm_male,spoof_score_cm_male,Prior_spoof,target_scores_male,nontarget_scores_male,spoof_scores_male,[asv_score_male],type)


_ ,_,_  = compute_t_dcf(bonafide_score_cm_female,spoof_score_cm_female,Prior_spoof,target_scores_female,nontarget_scores_female,spoof_scores_female,[asv_score_female],type)


The t-DCF evaluation type is: constrained
t-DCF evaluation from [Nbona=2288, Nspoof=20331] trials

t-DCF MODEL
   Ptar         =  0.94050 (Prior probability of target user)
   Pnon         =  0.00950 (Prior probability of nontarget user)
   Pspoof       =  0.05000 (Prior probability of spoofing attack)
   Cfa_asv      = 10.00000 (Cost of ASV falsely accepting a nontarget)
   Cmiss_asv    =  1.00000 (Cost of ASV falsely rejecting target speaker)
   Cfa_cm       = 10.00000 (Cost of CM falsely passing a spoof to ASV system)
   Cmiss_cm     =  1.00000 (Cost of CM falsely blocking target utterance which never reaches ASV)

   Implied normalized t-DCF function (depends on t-DCF parameters and ASV errors), s=CM threshold)
   tDCF_norm(s) =  2.82749 x Pmiss_cm(s) + Pfa_cm(s)

the asv threshold is from eer on asv development set  0.352
the CM thresholds is:  [4.97268677e-04 1.49726868e-03 1.50895119e-03 ... 9.96871948e-01
 9.96981561e-01 9.97154653e-01]
the CM threshold min is: 0.90961152315139

In [29]:
def metric(labels, scores):
    print_cost = False
    Prior_spoof = 0.05
    
    asv_score = 0.397
    bonafide_score_cm_male =  scores[labels == 0]
    spoof_score_cm_male = scores[labels == 1]
    bonafide_score_cm_female = scores[labels == 2]
    spoof_score_cm_female = scores[labels == 3]
    cost_model = {
            'Ptar':(1-Prior_spoof)*0.99,         # Prior probability of target speaker
            'Pnon':(1-Prior_spoof)*0.01,         # Prior probability of nontarget speaker (zero-effort impostor)
            'Pspoof': Prior_spoof,        # Prior probability of spoofing attack
            'Cmiss_asv': 1,      # Cost of ASV falsely rejecting target
            'Cfa_asv': 10,       # Cost of ASV falsely accepting nontarget
            'Cmiss_cm': 1,       # Cost of CM falsely rejecting target
            'Cfa_cm': 10          # Cost of CM falsely accepting spoof
        }
    tDCF_norm, tDCF, P_cm_miss_total, P_cm_fa_total, thresholds_male, thresholds_female = calc_t_dcf_spesific(target_scores_male,nontarget_scores_male,spoof_scores_male,spoof_score_cm_male,bonafide_score_cm_male,asv_score_male,[thr_male],
                                                                                                                    target_scores_female,nontarget_scores_female,spoof_scores_female,spoof_score_cm_female,bonafide_score_cm_female,asv_score_female,[thr_female],type_overlap = 'spesific')

    return min(tDCF_norm)


#tDCF_norm, tDCF, P_cm_miss_total, P_cm_fa_total, thresholds_male, thresholds_female = calc_t_dcf_spesific(target_scores_male,nontarget_scores_male,spoof_scores_male,spoof_score_cm_male,bonafide_score_cm_male,asv_score_male,[thr_male],
#                                                                                                                    target_scores_female,nontarget_scores_female,spoof_scores_female,spoof_score_cm_female,bonafide_score_cm_female,asv_score_female,[thr_female],type_overlap = 'all')

speakers = Eval_dataset_all.speaker_id
scores = np.concatenate((bonafide_score_cm_male,spoof_score_cm_male,bonafide_score_cm_female,spoof_score_cm_female))
labels = np.concatenate((np.zeros(len(bonafide_score_cm_male)),np.ones(len(spoof_score_cm_male)),np.ones(len(bonafide_score_cm_female)) *2 ,np.ones(len(spoof_score_cm_female)) * 3))

res_eval_preds = evaluate_with_conf_int(samples = scores, metric = metric, labels = labels, conditions = speakers, num_bootstraps=num_bootstraps, alpha=alpha)

print(res_eval_preds)


#with open('results_total_tdcf_wth_preds.pkl', 'wb') as f:
    #pickle.dump(results_total_tdcf_wth_preds, f)

(array([0.2709862], dtype=float32), (0.2499151635915041, 0.2923269957304001))


# Dev

In [30]:
labels_dev_male = torch.Tensor(Dev_dataset_all.is_spoofed[(Dev_dataset_all.sex == 'male').values].values).cpu().numpy().copy()
total_data = torch.Tensor(Dev_dataset_all.data[(Dev_dataset_all.sex == 'male').values]).cpu().numpy().copy()
with torch.no_grad():
    model = model.to(device)
    score_dev_male = torch.sigmoid(spoof_model_male(torch.Tensor(total_data).to(device)))
    
score_dev_male = score_dev_male.cpu().numpy().copy()

dev_male_eer, _ = my_functions.compute_eer(labels_dev_male,score_dev_male) # compute equal error rate

print(f"\tDev male EER: ({100*dev_male_eer}%)")


	Dev male EER: (0.5546536796531659%)


In [31]:
labels_dev_female = torch.Tensor(Dev_dataset_all.is_spoofed[(Dev_dataset_all.sex == 'female').values].values).cpu().numpy().copy()
total_data = torch.Tensor(Dev_dataset_all.data[(Dev_dataset_all.sex == 'female').values]).cpu().numpy().copy()
with torch.no_grad():
    model = model.to(device)
    output = spoof_model_female(torch.Tensor(total_data).to(device))
    _ , score_dev_female = spoof_model_female.loss(torch.Tensor(output).to(device),None)
    score_dev_female = -1*score_dev_female
    
score_dev_female = score_dev_female.cpu().numpy().copy()

eer, thresh = my_functions.compute_eer(labels_dev_female,score_dev_female) # compute equal error rate

print(f"\tDev Female EER:({100*eer}%)")

	Dev Female EER:(0.0%)


In [32]:
from ASV_utils.tdcf_functions import compute_t_dcf


import pickle
with open('objects_asv_and_cm_time_embeddings_avg_score_both_asv_protocol.pkl', 'rb') as f:
    results = pickle.load(f)

# Dev with Labels:

In [33]:
bonafide_score_cm_male_dev = 1 - score_dev_male[labels_dev_male==0].flatten()
spoof_score_cm_male_dev = 1 - score_dev_male[labels_dev_male==1].flatten()

target_scores_male_dev = results.loc[(results['asv_label'] ==  "target") & (results['its_male_ground_truth'] == 'male') ]["asv_score_male"].values
nontarget_scores_male_dev = results.loc[(results['asv_label'] == "nontarget") & (results['its_male_ground_truth'] == 'male') ]["asv_score_male"].values
spoof_scores_male_dev = results.loc[(results['asv_label'] ==  "spoof") & (results['its_male_ground_truth'] == 'male')]["asv_score_male"].values
list_asv_score_male = [0.261]



bonafide_score_cm_female_dev = score_dev_female[labels_dev_female==0]
bonafide_score_cm_female_dev = -1*bonafide_score_cm_female_dev
spoof_score_cm_female_dev = score_dev_female[labels_dev_female==1]
spoof_score_cm_female_dev = -1*spoof_score_cm_female_dev

target_scores_female_dev = results.loc[(results['asv_label'] ==  "target") & (results['its_male_ground_truth'] == 'female') ]["asv_score_female"].values
nontarget_scores_female_dev = results.loc[(results['asv_label'] == "nontarget") & (results['its_male_ground_truth'] == 'female') ]["asv_score_female"].values
spoof_scores_female_dev = results.loc[(results['asv_label'] ==  "spoof") & (results['its_male_ground_truth'] == 'female')]["asv_score_female"].values
list_asv_score_female = [0.397]


In [ ]:
#asv male
target_scores_male = target_scores_male_dev
nontarget_scores_male = nontarget_scores_male_dev
spoof_scores_male = spoof_scores_male_dev 
asv_score_male = list_asv_score_male[0]
# cm male
bonafide_score_cm_male = bonafide_score_cm_male_dev
spoof_score_cm_male = spoof_score_cm_male_dev
thr_male = 0.7791557312011719

#asv female
target_scores_female = target_scores_female_dev 
nontarget_scores_female = nontarget_scores_female_dev
spoof_scores_female = spoof_scores_female_dev 
asv_score_female = list_asv_score_female[0]
# cm female
bonafide_score_cm_female = bonafide_score_cm_female_dev
spoof_score_cm_female = spoof_score_cm_female_dev
thr_female = 0.546097993850708


_ ,_,_  = compute_t_dcf(bonafide_score_cm_male,spoof_score_cm_male,Prior_spoof,target_scores_male,nontarget_scores_male,spoof_scores_male,[asv_score_male],type)


_ ,_,_  = compute_t_dcf(bonafide_score_cm_female,spoof_score_cm_female,Prior_spoof,target_scores_female,nontarget_scores_female,spoof_scores_female,[asv_score_female],type)


The t-DCF evaluation type is: constrained
t-DCF evaluation from [Nbona=868, Nspoof=7392] trials

t-DCF MODEL
   Ptar         =  0.94050 (Prior probability of target user)
   Pnon         =  0.00950 (Prior probability of nontarget user)
   Pspoof       =  0.05000 (Prior probability of spoofing attack)
   Cfa_asv      = 10.00000 (Cost of ASV falsely accepting a nontarget)
   Cmiss_asv    =  1.00000 (Cost of ASV falsely rejecting target speaker)
   Cfa_cm       = 10.00000 (Cost of CM falsely passing a spoof to ASV system)
   Cmiss_cm     =  1.00000 (Cost of CM falsely blocking target utterance which never reaches ASV)

   Implied normalized t-DCF function (depends on t-DCF parameters and ASV errors), s=CM threshold)
   tDCF_norm(s) =  3.08466 x Pmiss_cm(s) + Pfa_cm(s)

the asv threshold is from eer on asv development set  0.261
the CM thresholds is:  [5.91026783e-04 1.59102678e-03 1.60676241e-03 ... 9.96470869e-01
 9.96529162e-01 9.97042716e-01]
the CM threshold min is: 0.7791557312011719

In [35]:
def metric(labels, scores):
    print_cost = False
    Prior_spoof = 0.05
    
    bonafide_score_cm_male =  scores[labels == 0]
    spoof_score_cm_male = scores[labels == 1]
    bonafide_score_cm_female = scores[labels == 2]
    spoof_score_cm_female = scores[labels == 3]
    cost_model = {
            'Ptar':(1-Prior_spoof)*0.99,         # Prior probability of target speaker
            'Pnon':(1-Prior_spoof)*0.01,         # Prior probability of nontarget speaker (zero-effort impostor)
            'Pspoof': Prior_spoof,        # Prior probability of spoofing attack
            'Cmiss_asv': 1,      # Cost of ASV falsely rejecting target
            'Cfa_asv': 10,       # Cost of ASV falsely accepting nontarget
            'Cmiss_cm': 1,       # Cost of CM falsely rejecting target
            'Cfa_cm': 10          # Cost of CM falsely accepting spoof
        }
    tDCF_norm, tDCF, P_cm_miss_total, P_cm_fa_total, thresholds_male, thresholds_female = calc_t_dcf_spesific(target_scores_male,nontarget_scores_male,spoof_scores_male,spoof_score_cm_male,bonafide_score_cm_male,asv_score_male,[thr_male],
                                                                                                                    target_scores_female,nontarget_scores_female,spoof_scores_female,spoof_score_cm_female,bonafide_score_cm_female,asv_score_female,[thr_female],type_overlap = 'spesific')

    return min(tDCF_norm)


#tDCF_norm, tDCF, P_cm_miss_total, P_cm_fa_total, thresholds_male, thresholds_female = calc_t_dcf_spesific(target_scores_male,nontarget_scores_male,spoof_scores_male,spoof_score_cm_male,bonafide_score_cm_male,asv_score_male,[thr_male],
#                                                                                                                    target_scores_female,nontarget_scores_female,spoof_scores_female,spoof_score_cm_female,bonafide_score_cm_female,asv_score_female,[thr_female],type_overlap = 'all')

speakers = Dev_dataset_all.speaker_id
scores = np.concatenate((bonafide_score_cm_male,spoof_score_cm_male,bonafide_score_cm_female,spoof_score_cm_female))
labels = np.concatenate((np.zeros(len(bonafide_score_cm_male)),np.ones(len(spoof_score_cm_male)),np.ones(len(bonafide_score_cm_female)) *2 ,np.ones(len(spoof_score_cm_female)) * 3))

res_dev_preds = evaluate_with_conf_int(samples = scores, metric = metric, labels = labels, conditions = speakers, num_bootstraps=num_bootstraps, alpha=alpha)

print(res_dev_preds)

(array([0.00398485], dtype=float32), (0.000503346607729327, 0.012481486354954533))


In [36]:
# tDCF_norm, tDCF, P_cm_miss_total, P_cm_fa_total, thresholds_male, thresholds_female = calc_t_dcf_spesific(target_scores_male,nontarget_scores_male,spoof_scores_male,spoof_score_cm_male,bonafide_score_cm_male,asv_score_male,[thr_male],
#                                                                                                                     target_scores_female,nontarget_scores_female,spoof_scores_female,spoof_score_cm_female,bonafide_score_cm_female,asv_score_female,[thr_female],type_overlap = 'spesific')

# #tDCF_norm, tDCF, P_cm_miss_total, P_cm_fa_total, thresholds_male, thresholds_female = calc_t_dcf_spesific(target_scores_male,nontarget_scores_male,spoof_scores_male,spoof_score_cm_male,bonafide_score_cm_male,asv_score_male,[thr_male],
# #                                                                                                                    target_scores_female,nontarget_scores_female,spoof_scores_female,spoof_score_cm_female,bonafide_score_cm_female,asv_score_female,[thr_female],type_overlap = 'all')

# results_total_tdcf_wth_labels_dev = {
#     'tDCF_norm': tDCF_norm,
#     'tDCF': tDCF,
#     'P_cm_miss_total': P_cm_miss_total,
#     'P_cm_fa_total': P_cm_fa_total,
#     'thresholds_male' : thresholds_male,
#     'thresholds_female': thresholds_female
# }


#with open('results_total_tdcf_wth_labels_dev.pkl', 'wb') as f:
#    pickle.dump(results_total_tdcf_wth_labels_dev, f)

# Dev with Preds:

In [37]:
labels_dev_male_gender_classification = torch.Tensor(Dev_dataset_all.is_spoofed[gender_model.predict(Dev_dataset_all.data_for_gender_classification) == 1]).cpu().numpy().copy()
total_data = torch.Tensor(Dev_dataset_all.data[gender_model.predict(Dev_dataset_all.data_for_gender_classification) == 1]).cpu().numpy().copy()
with torch.no_grad():
    model = model.to(device)
    score_dev_male_gender_classification = torch.sigmoid(spoof_model_male(torch.Tensor(total_data).to(device)))
    
score_dev_male_gender_classification = score_dev_male_gender_classification.cpu().numpy().copy()

dev_male_eer, _ = my_functions.compute_eer(labels_dev_male_gender_classification,score_dev_male_gender_classification) # compute equal error rate

print(f"\tDev male EER: ({100*dev_male_eer}%)")



	Dev male EER: (0.5546536796531659%)


In [38]:
labels_dev_female_gender_classification = torch.Tensor(Dev_dataset_all.is_spoofed[gender_model.predict(Dev_dataset_all.data_for_gender_classification) == 0].values).cpu().numpy().copy()
total_data = torch.Tensor(Dev_dataset_all.data[gender_model.predict(Dev_dataset_all.data_for_gender_classification) == 0]).cpu().numpy().copy()
with torch.no_grad():
    model = model.to(device)
    output = spoof_model_female(torch.Tensor(total_data).to(device))
    _ , score_dev_female_gender_classification = spoof_model_female.loss(torch.Tensor(output).to(device),None)
    score_dev_female_gender_classification = -1*score_dev_female_gender_classification
    
score_dev_female_gender_classification = score_dev_female_gender_classification.cpu().numpy().copy()

eer, thresh = my_functions.compute_eer(labels_dev_female_gender_classification,score_dev_female_gender_classification) # compute equal error rate

print(f"\tDev Female EER:({100*eer}%)")

	Dev Female EER:(0.0%)


In [39]:
bonafide_score_cm_male_dev = 1-score_dev_male_gender_classification[labels_dev_male_gender_classification==0].flatten()
spoof_score_cm_male_dev = 1-score_dev_male_gender_classification[labels_dev_male_gender_classification==1].flatten()
Prior_spoof = 0.05
target_scores_male_dev = results.loc[(results['asv_label'] ==  "target") & (results['its_male'] == True) ]["asv_score_male"].values
nontarget_scores_male_dev = results.loc[(results['asv_label'] == "nontarget") & (results['its_male'] == True) ]["asv_score_male"].values
spoof_scores_male_dev = results.loc[(results['asv_label'] ==  "spoof") & (results['its_male'] == True)]["asv_score_male"].values



bonafide_score_cm_female_dev = score_dev_female_gender_classification[labels_dev_female_gender_classification==0]
bonafide_score_cm_female_dev = -1*bonafide_score_cm_female_dev

spoof_score_cm_female_dev = score_dev_female_gender_classification[labels_dev_female_gender_classification==1]
spoof_score_cm_female_dev = -1*spoof_score_cm_female_dev

target_scores_female_dev = results.loc[(results['asv_label'] ==  "target") & (results['its_male'] == False) ]["asv_score_female"].values
nontarget_scores_female_dev = results.loc[(results['asv_label'] == "nontarget") & (results['its_male'] == False) ]["asv_score_female"].values
spoof_scores_female_dev = results.loc[(results['asv_label'] ==  "spoof") & (results['its_male'] == False)]["asv_score_female"].values


list_asv_score_female = [0.397]
list_asv_score_male = [0.261]

In [ ]:
#asv male
target_scores_male = target_scores_male_dev
nontarget_scores_male = nontarget_scores_male_dev
spoof_scores_male = spoof_scores_male_dev 
asv_score_male = list_asv_score_male[0]
# cm male
bonafide_score_cm_male = bonafide_score_cm_male_dev
spoof_score_cm_male = spoof_score_cm_male_dev
thr_male =    0.7791557312011719

#asv female
target_scores_female = target_scores_female_dev 
nontarget_scores_female = nontarget_scores_female_dev
spoof_scores_female = spoof_scores_female_dev 
asv_score_female = list_asv_score_female[0]
#
bonafide_score_cm_female = bonafide_score_cm_female_dev
spoof_score_cm_female = spoof_score_cm_female_dev
thr_female = 0.546097993850708

_ ,_,_  = compute_t_dcf(bonafide_score_cm_male,spoof_score_cm_male,Prior_spoof,target_scores_male,nontarget_scores_male,spoof_scores_male,[asv_score_male],type)


_ ,_,_  = compute_t_dcf(bonafide_score_cm_female,spoof_score_cm_female,Prior_spoof,target_scores_female,nontarget_scores_female,spoof_scores_female,[asv_score_female],type)

The t-DCF evaluation type is: constrained
t-DCF evaluation from [Nbona=868, Nspoof=7392] trials

t-DCF MODEL
   Ptar         =  0.94050 (Prior probability of target user)
   Pnon         =  0.00950 (Prior probability of nontarget user)
   Pspoof       =  0.05000 (Prior probability of spoofing attack)
   Cfa_asv      = 10.00000 (Cost of ASV falsely accepting a nontarget)
   Cmiss_asv    =  1.00000 (Cost of ASV falsely rejecting target speaker)
   Cfa_cm       = 10.00000 (Cost of CM falsely passing a spoof to ASV system)
   Cmiss_cm     =  1.00000 (Cost of CM falsely blocking target utterance which never reaches ASV)

   Implied normalized t-DCF function (depends on t-DCF parameters and ASV errors), s=CM threshold)
   tDCF_norm(s) =  3.08466 x Pmiss_cm(s) + Pfa_cm(s)

the asv threshold is from eer on asv development set  0.261
the CM thresholds is:  [5.91026783e-04 1.59102678e-03 1.60676241e-03 ... 9.96470869e-01
 9.96529162e-01 9.97042716e-01]
the CM threshold min is: 0.7791557312011719

In [41]:
tDCF_norm, tDCF, P_cm_miss_total, P_cm_fa_total, thresholds_male, thresholds_female = calc_t_dcf_spesific(target_scores_male,nontarget_scores_male,spoof_scores_male,spoof_score_cm_male,bonafide_score_cm_male,asv_score_male,[thr_male],
                                                                                                                    target_scores_female,nontarget_scores_female,spoof_scores_female,spoof_score_cm_female,bonafide_score_cm_female,asv_score_female,[thr_female],type_overlap = 'spesific')

#tDCF_norm, tDCF, P_cm_miss_total, P_cm_fa_total, thresholds_male, thresholds_female = calc_t_dcf_spesific(target_scores_male,nontarget_scores_male,spoof_scores_male,spoof_score_cm_male,bonafide_score_cm_male,asv_score_male,[thr_male],
#                                                                                                                   target_scores_female,nontarget_scores_female,spoof_scores_female,spoof_score_cm_female,bonafide_score_cm_female,asv_score_female,[thr_female],type_overlap = 'all')


results_total_tdcf_wth_preds_dev = {
    'tDCF_norm': tDCF_norm,
    'tDCF': tDCF,
    'P_cm_miss_total': P_cm_miss_total,
    'P_cm_fa_total': P_cm_fa_total,
    'thresholds_male' : thresholds_male,
    'thresholds_female': thresholds_female
}


#with open('results_total_tdcf_wth_preds_dev.pkl', 'wb') as f:
#    pickle.dump(results_total_tdcf_wth_preds_dev, f)

In [42]:
results_total_tdcf_wth_preds_dev

{'tDCF_norm': array([[0.00398485]], dtype=float32),
 'tDCF': array([[0.00088201]], dtype=float32),
 'P_cm_miss_total': array([[0.00039246]], dtype=float32),
 'P_cm_fa_total': array([[0.00233226]], dtype=float32),
 'thresholds_male': [0.7791557312011719],
 'thresholds_female': [0.546097993850708]}